# Fusion Perception Pipeline — Colab Setup (KITTI `2011_10_03_drive_0047`)

Uses the same KITTI raw sequence as `1_kitti_object_detection_lidar.ipynb`.
Run cells in order. Requires **T4 GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ── Cell 1: Mount Google Drive and navigate to project ────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Adjust this path to where Fusion_sensor lives in your Drive
PROJECT_DIR = '/content/drive/MyDrive/Fusion_sensor'   # ← edit if needed

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 2: Install PyTorch (CUDA 12.1 — matches Colab T4) ───────────────────
# Must install torch before everything else so the GPU build is picked up
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121 -q

# ── Project dependencies (from requirements.txt) ──────────────────────────────
!pip install \
    "omegaconf>=2.3.0" \
    "opencv-python>=4.9.0" \
    "Pillow>=10.0.0" \
    "numpy>=1.26.0" \
    "dataclasses-json>=0.6.0" \
    "rich>=13.0.0" \
    "huggingface_hub>=0.26.0" \
    "transformers>=4.47.0" \
    "bitsandbytes>=0.44.0" \
    "accelerate>=1.2.0" -q

# ── External model libraries ──────────────────────────────────────────────────
!pip install git+https://github.com/allenai/WildDet3D -q
!pip install git+https://github.com/facebookresearch/co-tracker -q

# ── Install this package in editable mode ─────────────────────────────────────
!pip install -e . -q

print('Installation complete.')

In [ ]:
# ── Cell 2: Download WildDet3D checkpoint from HuggingFace ───────────────────
!mkdir -p ckpt
!huggingface-cli download allenai/WildDet3D \
  wilddet3d_alldata_all_prompt_v1.0.pt --local-dir ckpt/

In [ ]:
# ── Cell 3: Download KITTI raw data (same sequence as 1_kitti_object_detection_lidar.ipynb)
import os

# Skip if already downloaded (e.g. persisted on Drive)
if not os.path.exists('2011_10_03/2011_10_03_drive_0047_sync'):
    !wget -q https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_10_03_drive_0047/2011_10_03_drive_0047_sync.zip
    !wget -q https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_10_03_calib.zip
    !jar xf 2011_10_03_drive_0047_sync.zip
    !jar xf 2011_10_03_calib.zip
    print('Download complete.')
else:
    print('Data already present — skipping download.')

In [ ]:
# ── Cell 4: Parse KITTI calibration → extract real K (3×3 intrinsics) ────────
import numpy as np
from glob import glob
import pandas as pd

DATA_PATH = '2011_10_03/2011_10_03_drive_0047_sync'
CALIB_PATH = '2011_10_03'

# Parse camera-to-camera calibration (same as original notebook)
with open(os.path.join(CALIB_PATH, 'calib_cam_to_cam.txt'), 'r') as f:
    calib = f.readlines()

# P_rect2_cam2: 3×4 projection matrix for left RGB camera
P_rect2_cam2 = np.array(
    [float(x) for x in calib[25].strip().split(' ')[1:]]
).reshape((3, 4))

# Extract 3×3 intrinsics K from the 3×4 projection matrix
# P = [K | Kt] where t is the translation; K is the top-left 3×3 block
K = P_rect2_cam2[:3, :3].astype(np.float32)

print('P_rect2_cam2 (3×4):')
print(P_rect2_cam2)
print()
print('K (3×3 intrinsics):')
print(K)
print()
print(f'  fx={K[0,0]:.2f}  fy={K[1,1]:.2f}  cx={K[0,2]:.2f}  cy={K[1,2]:.2f}')

In [ ]:
# ── Cell 5: Collect image paths and compute real FPS from timestamps ──────────
left_image_paths = sorted(glob(os.path.join(DATA_PATH, 'image_02/data/*.png')))
print(f'Found {len(left_image_paths)} left camera frames')

# Compute FPS from timestamps (same method as original notebook)
timestamps = pd.read_csv(
    os.path.join(DATA_PATH, 'image_02/timestamps.txt'),
    header=None
).squeeze().astype(object).apply(lambda x: x.split(' ')[1])

def hms_to_seconds(t):
    h, m, s = t.split(':')
    return int(h)*3600 + int(m)*60 + float(s)

total_seconds = timestamps.apply(hms_to_seconds)
KITTI_FPS = float(1.0 / np.median(np.diff(total_seconds.values)))
print(f'KITTI camera FPS: {KITTI_FPS:.2f}')

In [ ]:
# ── Cell 6: Convert KITTI PNG sequence → MP4 for VideoLoader ─────────────────
import cv2

# Optional: limit to first N frames for a quick test run
MAX_FRAMES = None   # set to e.g. 150 for ~10s; None = full sequence

paths_to_encode = left_image_paths if MAX_FRAMES is None else left_image_paths[:MAX_FRAMES]
sample = cv2.imread(paths_to_encode[0])
h, w = sample.shape[:2]

VIDEO_PATH = '/content/kitti_0047.mp4'
writer = cv2.VideoWriter(
    VIDEO_PATH,
    cv2.VideoWriter_fourcc(*'mp4v'),
    KITTI_FPS,
    (w, h)
)
for p in paths_to_encode:
    writer.write(cv2.imread(p))   # already BGR — VideoWriter expects BGR
writer.release()

print(f'Encoded {len(paths_to_encode)} frames → {VIDEO_PATH}')
print(f'Resolution: {w}×{h}  FPS: {KITTI_FPS:.2f}')

In [ ]:
# ── Cell 7: Verify GPU ────────────────────────────────────────────────────────
import torch

assert torch.cuda.is_available(), 'No GPU — change runtime to T4'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {vram_gb:.1f} GB')
assert vram_gb >= 10.0, f'Need ≥10 GB VRAM, got {vram_gb:.1f} GB'
print('GPU check passed')

In [ ]:
# ── Cell 8: Smoke test (pure-Python, no GPU needed) ───────────────────────────
from fusion_perception.utils.dataclasses import Track, OccupancyGrid, SceneMemory
from fusion_perception.utils.logging_setup import setup_logging, get_logger
from fusion_perception.occupancy.bev_grid import OccupancyBEVGenerator
from fusion_perception.reasoning.prompt_builder import build_scene_prompt

setup_logging(level='INFO', use_rich=False)
logger = get_logger('smoke_test')

# BEV grid
bev = OccupancyBEVGenerator(resolution=0.5, x_range=[-20.,20.], z_range=[0.,50.], decay_factor=0.95)
grid = bev.update([], frame_idx=0)
assert len(grid.grid) == 100
logger.info('BEV grid OK')

# Prompt builder with a mock track
track = Track(
    track_id=1, class_name='car', first_seen=0, last_seen=5,
    centroid_history=[[320.,240.],[325.,238.]],
    position_3d_history=[[0.,0.,15.],[0.,0.,14.]],
    cow_query_point=[325.,238.], is_active=True, occlusion_count=0,
)
occ = OccupancyGrid(frame_idx=5, resolution=0.5, x_range=[-20.,20.], z_range=[0.,50.],
                    grid=[[0.]*80 for _ in range(100)], decay_factor=0.95)
mem = SceneMemory(frame_idx=5, active_tracks=[track], occupancy_grid=occ,
                  event_flags=['new_object:1'], frame_count=5, elapsed_seconds=0.17)
prompt = build_scene_prompt(mem)
assert 'car' in prompt
logger.info('Prompt builder OK')

# K matrix from Cell 4 is a real 3×3 float32 array
assert K.shape == (3, 3) and K.dtype == np.float32
logger.info(f'K matrix OK: fx={K[0,0]:.1f}')

print('\n=== All smoke tests passed ===')

In [ ]:
# ── Cell 9: Run pipeline on KITTI sequence with real intrinsics ───────────────
#
# Uses a custom frame loop instead of StreamingPipeline.run() so we can
# pass the real KITTI K matrix (parsed in Cell 4) to every detect() call.

from omegaconf import OmegaConf
from pathlib import Path
import cv2, datetime

from fusion_perception.models.wilddet3d_wrapper import WildDet3DWrapper
from fusion_perception.tracking.cowtracker_wrapper import CoWTrackerWrapper
from fusion_perception.occupancy.bev_grid import OccupancyBEVGenerator
from fusion_perception.tracking.trajectory_manager import SceneMemoryManager
from fusion_perception.models.gemma_wrapper import GemmaReasoningWrapper
from fusion_perception.pipelines.stage_runner import StageRunner
from fusion_perception.utils.video_loader import VideoLoader
from fusion_perception.utils.json_io import save_detections, save_tracks, save_reasoning
from fusion_perception.utils.memory_monitor import log_gpu_memory

# ── Config ────────────────────────────────────────────────────────────────────
cfg = OmegaConf.merge(
    OmegaConf.load('configs/default.yaml'),
    OmegaConf.load('configs/colab.yaml'),
)
cfg.video.source = VIDEO_PATH
cfg.video.max_frames = None          # process full sequence; set e.g. 150 for a quick test
cfg.video.resize_hw = [h, w]         # KITTI native resolution (370×1224)
cfg.detection.prompts = ['car', 'person', 'cyclist', 'van', 'truck']

run_id = datetime.datetime.now().strftime('%Y-%m-%d_%H%M%S') + '_kitti_0047'
out_dir = Path(cfg.output.base_dir) / run_id
out_dir.mkdir(parents=True, exist_ok=True)

# ── Load models ───────────────────────────────────────────────────────────────
detector = WildDet3DWrapper(
    score_threshold=cfg.detection.score_threshold,
    fp16=cfg.detection.fp16,
)
detector.load(cfg.detection.checkpoint, cfg.detection.device)

tracker = CoWTrackerWrapper(
    window_size=cfg.tracking.window_size,
    max_tracks=cfg.tracking.max_tracks,
    occlusion_tolerance=cfg.tracking.occlusion_tolerance,
    nn_threshold=cfg.tracking.nn_threshold,
    device=cfg.tracking.device,
)
tracker.load()

bev_gen = OccupancyBEVGenerator(
    resolution=cfg.occupancy.resolution,
    x_range=list(cfg.occupancy.x_range),
    z_range=list(cfg.occupancy.z_range),
    decay_factor=cfg.occupancy.decay_factor,
)

scene_mem = SceneMemoryManager()

gemma = GemmaReasoningWrapper(
    model_id=cfg.reasoning.model_id,
    quantize_4bit=cfg.reasoning.quantize_4bit,
    max_new_tokens=cfg.reasoning.max_new_tokens,
    device=cfg.reasoning.device,
    visual_mode=cfg.reasoning.visual_mode,
)
if cfg.reasoning.enabled:
    gemma.load()

log_gpu_memory('All models loaded')

runner = StageRunner(
    detector=detector,
    tracker=tracker,
    bev_generator=bev_gen,
    scene_memory=scene_mem,
    gemma=gemma,
    prompts=list(cfg.detection.prompts),
    reasoning_interval=cfg.reasoning.interval_frames,
    fps=KITTI_FPS,
    visual_reasoning=cfg.reasoning.visual_mode,
)

# ── Frame loop — passes real KITTI K to every detect() call ──────────────────
loader = VideoLoader(
    source=VIDEO_PATH,
    resize_hw=None,              # already at native resolution
    max_frames=cfg.video.max_frames,
)

video_writer = cv2.VideoWriter(
    str(out_dir / 'output.mp4'),
    cv2.VideoWriter_fourcc(*'mp4v'),
    KITTI_FPS,
    (w + h + 200, h),           # frame + BEV + text panel
)

all_detections, all_tracks, all_reasoning = {}, {}, []
FLUSH_EVERY = cfg.output.flush_interval

for frame_idx, frame_rgb, meta in loader:
    # Pass real KITTI K — avoids the default auto-estimate
    outputs = runner.run_frame(frame_rgb, frame_idx, intrinsics=K)

    all_detections[frame_idx] = outputs['detections']
    for t in outputs['tracks']:
        all_tracks[t.track_id] = t
    if outputs['reasoning']:
        all_reasoning.append(outputs['reasoning'])

    bgr = outputs['composite'][:, :, ::-1]
    video_writer.write(bgr)

    if (frame_idx + 1) % FLUSH_EVERY == 0:
        save_detections(out_dir / 'detections.json', run_id,
                        VIDEO_PATH, list(cfg.detection.prompts), all_detections)
        save_tracks(out_dir / 'tracks.json', run_id, all_tracks)
        print(f'  Flushed at frame {frame_idx}')

# Final flush + close
save_detections(out_dir / 'detections.json', run_id,
                VIDEO_PATH, list(cfg.detection.prompts), all_detections)
save_tracks(out_dir / 'tracks.json', run_id, all_tracks)
if all_reasoning:
    save_reasoning(out_dir / 'reasoning.json', run_id,
                   cfg.reasoning.interval_frames, all_reasoning)
video_writer.release()

print(f'\nDone. {frame_idx+1} frames processed.')
print(f'Outputs → {out_dir}')

In [ ]:
# ── Cell 10: Preview a frame from the annotated output video ─────────────────
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (20, 6)

cap = cv2.VideoCapture(str(out_dir / 'output.mp4'))
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)   # jump to frame 30
ret, frame_bgr = cap.read()
cap.release()

if ret:
    plt.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    plt.title('Annotated output — frame 30 (annotated frame | BEV | reasoning panel)')
    plt.axis('off')
else:
    print('Could not read output video.')